<a href="https://colab.research.google.com/github/AnnaZapototska/colab-code-work/blob/development/homework_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1. Error log analytics

## Моделювання ймовірностей подій (DB / NET помилки)

За одну добу зафіксовано 500 подій. З них:

* 85 — лише помилки бази даних (DB)
* 60 — лише мережеві помилки (NET)
* 35 — помилки з обома тегами (DB і NET)

Позначимо:

* $\Omega$ — множина всіх подій ($|\Omega| = 500$)
* $A$ — подія "помилка бази даних" (DB)
* $B$ — подія "мережева помилка" (NET)

---

### 1. Математична інтерпретація

Загальна кількість подій з тегом DB:

$$
|A| = 85 + 35 = 120
$$

Загальна кількість подій з тегом NET:

$$
|B| = 60 + 35 = 95
$$

---

### 2. Ймовірність помилки DB

$$
P(A) = \frac{|A|}{|\Omega|} = \frac{120}{500}
$$

---

### 3. Ймовірність DB або NET

Використаємо формулу включень-виключень:

$$
P(A \cup B) = \frac{|A| + |B| - |A \cap B|}{|\Omega|}
$$

$$
P(A \cup B) = \frac{120 + 95 - 35}{500}
$$

---

### 4. Ймовірність DB без NET

$$
P(A \setminus B) = \frac{85}{500}
$$

### Висновок

* ймовірності обчислюються як відношення кількості подій до загальної кількості
* формула включень-виключень дозволяє уникнути подвійного рахунку
* частка "чистих" DB-помилок менша за загальну кількість DB-подій

In [14]:
def event_probabilities(total, only_db, only_net, both):
    # Total events with each tag
    db = only_db + both
    net = only_net + both

    # Probabilities
    p_db = db / total
    p_union = (db + net - both) / total
    p_db_only = only_db / total

    return p_db, p_union, p_db_only


if __name__ == "__main__":
    total = 500
    only_db = 85
    only_net = 60
    both = 35

    p_db, p_union, p_db_only = event_probabilities(total, only_db, only_net, both)

    print(f"Total events: {total}")
    print(f"\nP(DB error) = {p_db:.4f} ({p_db*100:.2f}%)")
    print(f"\nP(DB or NET) = {p_union:.4f} ({p_union*100:.2f}%)")
    print(f"\nP(DB only, not NET) = {p_db_only:.4f} ({p_db_only*100:.2f}%)")

Total events: 500

P(DB error) = 0.2400 (24.00%)

P(DB or NET) = 0.3600 (36.00%)

P(DB only, not NET) = 0.1700 (17.00%)


# Task 2. Battery quality control

## Моделювання ймовірностей вибору батарей

На складі зберігається 12 батарей:

* 8 — справні
* 4 — дефектні

Для тестування випадково обирають 3 батареї **без повернення**.

Позначимо:

* $G$ — кількість справних батарей
* $B$ — кількість дефектних батарей
* $k$ — кількість вибраних батарей

---

### 1. Ймовірність, що всі 3 батареї справні (правило множення)

Ймовірність послідовного вибору:

$$
P = \frac{8}{12} \cdot \frac{7}{11} \cdot \frac{6}{10}
$$

Це правило умовних ймовірностей без повернення.

---

### 2. Ймовірність, що всі 3 батареї справні (комбінаторика)

$$
P = \frac{C(8,3)}{C(12,3)}
$$

де:

$$
C(n,k) = \frac{n!}{k!(n-k)!}
$$

---

### 3. Ймовірність, що рівно 2 батареї справні

#### Комбінаторний підхід:

$$
P = \frac{C(8,2)\cdot C(4,1)}{C(12,3)}
$$

---

#### Через правило множення:

Можливі варіанти:

* GGB
* GBG
* BGG

Наприклад:

$$
P(GGB) = \frac{8}{12} \cdot \frac{7}{11} \cdot \frac{4}{10}
$$

Загальна ймовірність:

$$
P = P(GGB) + P(GBG) + P(BGG)
$$

### Висновок

* ймовірності можна обчислювати різними методами
* комбінаторика є найзручнішою для таких задач
* модель відповідає гіпергеометричному розподілу


In [16]:
import math
from itertools import product

def C(n, k):
    return math.comb(n, k)

def all_good_multiplication(good, bad, k):
    total = good + bad
    p = 1.0
    for i in range(k):
        p *= (good - i) / (total - i)
    return p

def all_good_combinatorics(good, bad, k):
    total = good + bad
    return C(good, k) / C(total, k)

def exactly_r_good_combinatorics(good, bad, k, r):
    total = good + bad
    return (C(good, r) * C(bad, k - r)) / C(total, k)

def exactly_r_good_multiplication(good, bad, k, r):
    total = good + bad
    p = 0.0

    for seq in product([0, 1], repeat=k):
        if sum(seq) != r:
            continue

        g = good
        b = bad
        t = total
        prob = 1.0

        for step in seq:
            if step == 1:
                prob *= g / t
                g -= 1
            else:
                prob *= b / t
                b -= 1
            t -= 1

        p += prob

    return p


if __name__ == "__main__":
    good, bad, k = 8, 4, 3

    # All good
    p_all_mult = all_good_multiplication(good, bad, k)
    p_all_comb = all_good_combinatorics(good, bad, k)
    print("All 3 good (multiplication):", p_all_mult)
    print("All 3 good (combinatorics):", p_all_comb)

    # Just 2 good
    p_exact2_mult = exactly_r_good_multiplication(good, bad, k, r=2)
    p_exact2_comb = exactly_r_good_combinatorics(good, bad, k, r=2)


    print("\nExactly 2 good (multiplication):", p_exact2_mult)
    print("Exactly 2 good (combinatorics):", p_exact2_comb)

All 3 good (multiplication): 0.2545454545454545
All 3 good (combinatorics): 0.2545454545454545

Exactly 2 good (multiplication): 0.509090909090909
Exactly 2 good (combinatorics): 0.509090909090909


In [17]:
import random

def simulate(good, bad, k, trials=100000):
    success = 0

    for _ in range(trials):
        items = [1]*good + [0]*bad
        sample = random.sample(items, k)

        if sum(sample) == 2:
            success += 1

    return success / trials


print("Monte Carlo:", simulate(8, 4, 3))

Monte Carlo: 0.50919


# Task 3. Distributed system reliability

## Моделювання надійності розподіленої системи

Розглядається система з 4 незалежних вузлів. Ймовірності відмови протягом однієї години:

* Вузол A: $0.02$
* Вузол B: $0.05$
* Вузол C: $0.03$
* Вузол D: $0.04$

Позначимо:

* $A_1$ — відмова вузла A
* $A_2$ — відмова вузла B
* $A_3$ — відмова вузла C
* $A_4$ — відмова вузла D

---

### 1. Ймовірність, що всі вузли працюють

Ймовірність роботи вузла:

$$
P(\text{працює}) = 1 - p
$$

Тоді:

$$
P(\text{всі працюють}) = (1-0.02)(1-0.05)(1-0.03)(1-0.04)
$$

$$
P \approx 0.8669
$$

---

### 2. Ймовірність, що хоча б один вузол відмовить

Використаємо протилежну подію:

$$
P(\text{хоча б один}) = 1 - P(\text{всі працюють})
$$

$$
P \approx 1 - 0.8669 = 0.1331
$$

---

### 3. Ймовірність, що рівно два вузли відмовлять

Перебираємо всі можливі пари (6 варіантів):

$$
P = \sum P(\text{2 вузли не працюють, 2 працюють})
$$

Приклад для вузлів A і B:

$$
P(A,B) = 0.02 \cdot 0.05 \cdot (1-0.03) \cdot (1-0.04)
$$

Аналогічно для всіх пар:

$$
P \approx 0.00665
$$

---

### 4. Перевірка повноти розподілу

Сума ймовірностей усіх можливих випадків:

$$
P(0) + P(1) + P(2) + P(3) + P(4) = 1
$$

---

### Висновок

* використано незалежність подій
* застосовано правило множення та перебір комбінацій
* модель описує повний розподіл кількості відмов

In [15]:
from itertools import combinations

p_node = {
    "A": 0.02,
    "B": 0.05,
    "C": 0.03,
    "D": 0.04,
}

def p_all_work(p):
    prob = 1.0
    for name, pi in p.items():
        prob *= (1 - pi)
    return prob

def p_at_least_one_do_not_work(p):
    return 1 - p_all_work(p)

def p_exactly_k_do_not_work(p, k):
    nodes = list(p.keys())
    total = 0.0

    for late_set in combinations(nodes, k):
        late_set = set(late_set)
        prob = 1.0
        for node in nodes:
            if node in late_set:
                prob *= p[node]
            else:
                prob *= (1 - p[node])
        total += prob

    return total

if __name__ == "__main__":
    p_w = p_all_work(p_node)
    p_f = p_at_least_one_do_not_work(p_node)
    p_t = p_exactly_k_do_not_work(p_node, 2)

    print("All nodes are working during 1h", p_w, f"({p_w*100:.2f}%)")
    print("At least one is not working:", p_f, f"({p_f*100:.2f}%)")
    print("Exactly two nodes will fail:", p_t, f"({p_t*100:.2f}%)")

# check previous calculation

total_prob = 0.0
print("\n")
for k in range(5):  # від 0 до 4 вузлів
    pk = p_exactly_k_do_not_work(p_node, k)
    print(f"P({k} failures) = {pk:.6f}")
    total_prob += pk

print("Sum of all probabilities:", total_prob)

All nodes are working during 1h 0.8669471999999999 (86.69%)
At least one is not working: 0.13305280000000008 (13.31%)
Exactly two nodes will fail: 0.0066451999999999995 (0.66%)


P(0 failures) = 0.866947
P(1 failures) = 0.126257
P(2 failures) = 0.006645
P(3 failures) = 0.000149
P(4 failures) = 0.000001
Sum of all probabilities: 1.0


# Task 4. Traffic source identification

## Моделювання ймовірності покупки (формула повної ймовірності та теорема Баєса)

Інтернет-магазин отримує трафік з трьох джерел:

* Пошукова реклама — 50% трафіку
* Соцмережі — 30% трафіку
* Email-розсилка — 20% трафіку

Конверсія (ймовірність покупки):

* Пошукова реклама — 4%
* Соцмережі — 2%
* Email — 8%

Позначимо:

* $H_1$ — користувач прийшов з пошуку
* $H_2$ — користувач прийшов з соцмереж
* $H_3$ — користувач прийшов з email
* $A$ — користувач зробив покупку

---

### 1. Загальна ймовірність покупки

Використаємо формулу повної ймовірності:

$$
P(A) = \sum P(H_i)\cdot P(A \mid H_i)
$$

$$
P(A) = 0.5 \cdot 0.04 + 0.3 \cdot 0.02 + 0.2 \cdot 0.08
$$

$$
P(A) = 0.042
$$

---

### 2. Ймовірність, що покупець прийшов з email

Використаємо теорему Баєса:

$$
P(H_3 \mid A) = \frac{P(H_3)\cdot P(A \mid H_3)}{P(A)}
$$

$$
P(H_3 \mid A) = \frac{0.2 \cdot 0.08}{0.042}
$$

$$
P(H_3 \mid A) \approx 0.381
$$

---

### Висновок

* формула повної ймовірності дозволяє обчислити загальну ймовірність події
* теорема Баєса дозволяє знайти джерело події після її настання
* джерела з меншою часткою трафіку можуть мати більший вплив через високу конверсію


In [18]:

traffic = {
    "search": 0.5,
    "social": 0.3,
    "email": 0.2
}

conversion = {
    "search": 0.04,
    "social": 0.02,
    "email": 0.08
}

# 1

def total_probability(traffic, conversion):
    p = 0.0
    for source in traffic:
        p += traffic[source] * conversion[source]
    return p


p_A = total_probability(traffic, conversion)

print(f"Total conversion probability: {p_A:.4f} ({p_A*100:.2f}%)")

# 2

def buyer_email(traffic, conversion, target='email'):
    p_A = total_probability(traffic, conversion)
    return (traffic[target] * conversion[target]) / p_A

p_email = buyer_email(traffic, conversion)

print(f"\nP(email | purchase) = {p_email:.4f} ({p_email*100:.2f}%)")


def bayes_source(prior_probs, conversion_rates):
    p_A = sum(p * c for p, c in zip(prior_probs, conversion_rates))
    posterior = []
    for p, c in zip(prior_probs, conversion_rates):
        posterior.append((p * c) / p_A)

    return posterior

prior_probs = list(traffic.values())
conversion_rates = list(conversion.values())

result = bayes_source(prior_probs, conversion_rates)

print("Posterior probabilities:")
for i, p in enumerate(result):
    print(f"Source {i+1}: {p:.4f} ({p*100:.2f}%)")

# check calculation of probabilities
print("Sum:", sum(result))


Total conversion probability: 0.0420 (4.20%)

P(email | purchase) = 0.3810 (38.10%)
Posterior probabilities:
Source 1: 0.4762 (47.62%)
Source 2: 0.1429 (14.29%)
Source 3: 0.3810 (38.10%)
Sum: 1.0
